# **COMPARISON BETWEEN RANDOM FOREST AND DUMMY MODEL**

This document first builds a Random Forest model to predict recidivism. It then constructs a baseline model, which is a simple model that always predicts that an individual will not reoffend (the majority class). Finally, the two models are compared using the number of false positives as the main evaluation metric, as this is the metric we have chosen to assess model performance.

**DATA PREPARATION**

In [1]:
import pandas as pd
import numpy as np

# for imputing modes and medians
from sklearn.impute import SimpleImputer

# for random forest
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# for dummy model (baseline)
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report


In [2]:
target = 'is_recid'

In [3]:
dfOriginal = pd.read_csv("cox-violent-parsed_filt.csv")

In [4]:
dfProcessed = dfOriginal.drop_duplicates(subset=['name'])
dfProcessed.count()

,0
id,6560
name,10855
first,10855
last,10855
sex,10855
dob,10855
age,10855
age_cat,10855
race,10855
juv_fel_count,10855


In [5]:
dfProcessed = dfProcessed[['sex', 'age', 'race', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count', 'c_charge_degree', target]]
dfProcessed.count()

,0
sex,10855
age,10855
race,10855
juv_fel_count,10855
juv_misd_count,10855
juv_other_count,10855
priors_count,10855
c_charge_degree,10185
is_recid,10855


In [6]:
dfCheck = dfProcessed.loc[dfProcessed[target] < 0]
dfCheck.count()

,0
sex,648
age,648
race,648
juv_fel_count,648
juv_misd_count,648
juv_other_count,648
priors_count,648
c_charge_degree,1
is_recid,648


In [7]:
dfProcessed = dfProcessed.loc[dfProcessed[target] > -1]
dfProcessed.count()

,0
sex,10207
age,10207
race,10207
juv_fel_count,10207
juv_misd_count,10207
juv_other_count,10207
priors_count,10207
c_charge_degree,10184
is_recid,10207


In [8]:
print("Any NaN values?\n", dfProcessed.isnull().values.any())

Any NaN values?
 True


Here we can see that altough the heatmap looked like there were no missing values still there were.

Missing value strategy
1. Numerical values --> MEDIAN imputation
2. Categorical values --> MODE imputation

**RANDOM FOREST**

In [9]:
X = dfProcessed.drop(columns=[target])
y = dfProcessed[target]

testSize = 0.3
randNum = 44

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=testSize, random_state=randNum)

numCols = ['age', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count']

numImputer = SimpleImputer(strategy='median')
X_train[numCols] = numImputer.fit_transform(X_train[numCols])
X_test[numCols] = numImputer.transform(X_test[numCols])

catCols = ['sex', 'race', 'c_charge_degree']

catImputer = SimpleImputer(strategy='most_frequent')
X_train[catCols] = catImputer.fit_transform(X_train[catCols])
X_test[catCols] = catImputer.transform(X_test[catCols])

X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=randNum,
    min_samples_leaf=5,
    max_depth=10,
    class_weight='balanced'
)

rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10,
                       min_samples_leaf=5, n_estimators=300, random_state=44)

In [10]:
predictions = rf.predict(X_test)
predictions

array([0, 1, 0, ..., 1, 0, 0])

In [11]:
from sklearn.metrics import classification_report
print(classification_report(y_test, rf.predict(X_test)))

              precision    recall  f1-score   support

           0       0.79      0.66      0.72      2047
           1       0.49      0.65      0.56      1016

    accuracy                           0.66      3063
   macro avg       0.64      0.66      0.64      3063
weighted avg       0.69      0.66      0.67      3063



**BASELINE MODEL**

In [12]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

y_dummy_pred = dummy.predict(X_test)

print(classification_report(y_test, y_dummy_pred))

              precision    recall  f1-score   support

           0       0.67      1.00      0.80      2047
           1       0.00      0.00      0.00      1016

    accuracy                           0.67      3063
   macro avg       0.33      0.50      0.40      3063
weighted avg       0.45      0.67      0.54      3063



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**MODEL COMPARISON BY FPR**

In [13]:
df_eval = pd.DataFrame({
    "y_true": y_test,
    "race": dfProcessed.loc[y_test.index, "race"]
})

df_eval["rf_pred"] = rf.predict(X_test)
df_eval["dummy_pred"] = dummy.predict(X_test)

def false_positive_rate(y_true, y_pred):
    fp = np.sum((y_pred == 1) & (y_true == 0))
    tn = np.sum((y_pred == 0) & (y_true == 0))
    return fp / (fp + tn)

fpr_rf = false_positive_rate(df_eval["y_true"], df_eval["rf_pred"])
fpr_dummy = false_positive_rate(df_eval["y_true"], df_eval["dummy_pred"])

print(f"Random Forest FPR: {fpr_rf:.3f}")
print(f"Dummy Model FPR:   {fpr_dummy:.3f}")

Random Forest FPR: 0.335
Dummy Model FPR:   0.000


In [14]:
fpr_by_race = {}

for race in df_eval["race"].unique():
    subset = df_eval[df_eval["race"] == race]
    fpr_by_race[race] = {
        "RandomForest": false_positive_rate(
            subset["y_true"], subset["rf_pred"]
        ),
        "Dummy": false_positive_rate(
            subset["y_true"], subset["dummy_pred"]
        )
    }

pd.DataFrame(fpr_by_race).T

,RandomForest,Dummy
Hispanic,0.178571,0.0
Other,0.181102,0.0
Caucasian,0.247436,0.0
African-American,0.476615,0.0
Asian,0.000000,0.0
Native American,0.222222,0.0
